## enrichment

In [24]:
import pandas as pd
import numpy as np
import requests

## Read the JSON file that you saved in ex02

In [25]:
df = pd.read_json('../data/auto.json')
pd.options.display.float_format = '{:.2f}'.format

## Enrich the dataframe using a sample from that dataframe

In [26]:
np.random.seed(21)

sample_df = pd.DataFrame({
    'CarNumber': df.sample(n=200, random_state=21)['CarNumber'].values,
    'Make': df.sample(n=200, random_state=21)['Make'].values,
    'Model': df.sample(n=200, random_state=21)['Model'].values,
    'Fines': np.random.choice(df['Fines'], size=200),
    'Refund': np.random.choice(df['Refund'], size=200)
})
concat_rows = pd.concat([df, sample_df], ignore_index=True)

## Enrich the concat_rows dataframe with a new column containing generated data

In [27]:
np.random.seed(21)
Year = pd.Series(np.random.randint(1980, 2019, size=concat_rows.index.size), name='Year')
fines = pd.concat([concat_rows, Year], axis=1)

## Enrich the dataframe with data from another dataframe

In [28]:
most_popular_surnames = pd.read_json('../../datasets/surname.json', orient='values')
most_popular_surnames.columns = most_popular_surnames.iloc[0]
most_popular_surnames = most_popular_surnames[1:]

In [29]:
most_popular_surnames['NAME'] = (
    most_popular_surnames['NAME']
    .astype(str)
    .str.replace(r'[\[\]"]', '', regex=True)
    .str.strip()
)
unique_cars = concat_rows['CarNumber'].unique()

In [30]:
surnames = most_popular_surnames['NAME'].sample(
    n=len(unique_cars),
    replace=True,
    random_state=21
)

In [31]:
owners = pd.DataFrame({
    'CarNumber': unique_cars,
    'SURNAME': surnames.values
})

## Create a pivot table from the fines dataframe

In [32]:
new_rows = pd.DataFrame({
    'CarNumber': ['TEST001RUS', 'TEST002RUS', 'TEST003RUS', 'TEST004RUS', 'TEST005RUS'],
    'Make': ['Tesla', 'Chevrolet', 'Nissan', 'Hyundai', 'Mazda'],
    'Model': ['Model 3', 'Malibu', 'Altima', 'Elantra', 'CX-5'],
    'Fines': [0.0, 450.75, 120.50, 85.30, 210.00],
    'Refund': [50.00, 0.0, 25.00, 10.00, 35.00],
    'Year': [2022, 2016, 2019, 2021, 2018]
})

fines = pd.concat([fines, new_rows], ignore_index=True)
len(owners)

531

## Delete the last 20 observations from the owners dataframe and add three new observations that are not the same as those added to the fines dataframe.

In [33]:
owners = owners.iloc[:-20].reset_index(drop=True)
new_owners = pd.DataFrame({
    'CarNumber': ['TEST006RUS', 'TEST007RUS', 'TEST008RUS'],
    'SURNAME': ['MILLER', 'JEFFRI', 'JARVIS']
})
owners = pd.concat([owners, new_owners], ignore_index=True)

missing_cars = [car for car in fines['CarNumber'].unique()
                if car not in owners['CarNumber'].values
                and 'TEST' not in str(car)]
fix_df = pd.DataFrame({
    'CarNumber': missing_cars[:3],
    'SURNAME': ['SMITH'] * 3
})

owners = pd.concat([owners, fix_df], ignore_index=True)

## Join the two dataframes

## The new dataframe should contain only the car numbers that exist in both dataframes

In [34]:
inner_merge = pd.merge(fines, owners, on='CarNumber', how='inner')

## The new dataframe should contain all the car numbers from both dataframes

In [35]:
outer_merge = pd.merge(fines, owners, on='CarNumber', how='outer')

## The new dataframe should contain only the car numbers from the fines dataframe

In [36]:
left_merge = pd.merge(fines, owners, on='CarNumber', how='left')

## The new dataframe should contain only the car numbers from the owners dataframe

In [37]:
right_merge = pd.merge(fines, owners, on='CarNumber', how='right')

## Create a pivot table from the fines dataframe

In [38]:
fines.pivot_table(columns='Year', index=['Make', 'Model'], values='Fines', aggfunc='sum')

Year                   1980      1981     1982      1983      1984      1985  \
Make       Model                                                               
Chevrolet  Malibu       NaN       NaN      NaN       NaN       NaN       NaN   
Ford       Focus   74994.59 306394.59 88994.59 163994.59 112500.00 262694.59   
           Mondeo       NaN       NaN 46200.00       NaN       NaN       NaN   
Hyundai    Elantra      NaN       NaN      NaN       NaN       NaN       NaN   
Mazda      CX-5         NaN       NaN      NaN       NaN       NaN       NaN   
Nissan     Altima       NaN       NaN      NaN       NaN       NaN       NaN   
Skoda      Octavia  9094.59   1900.00  8894.59       NaN    500.00  10894.59   
Tesla      Model 3      NaN       NaN      NaN       NaN       NaN       NaN   
Toyota     Camry   12000.00       NaN 43600.00   1000.00   1000.00       NaN   
           Corolla      NaN   6800.00      NaN  12800.00       NaN   6300.00   
Volkswagen Golf    20800.00   8594.59  5000.00    200.00       NaN 168000.00   
           Jetta        NaN   1000.00      NaN       NaN       NaN   9000.00   
           Passat    900.00   4000.00      NaN   1100.00   8594.59       NaN   
           Touareg      NaN       NaN      NaN       NaN       NaN       NaN   

Year                   1986      1987     1988      1989  ...      2012  \
Make       Model                                          ...             
Chevrolet  Malibu       NaN       NaN      NaN       NaN  ...       NaN   
Ford       Focus   62200.00 161494.59 77772.93 168594.59  ... 143789.17   
           Mondeo       NaN       NaN      NaN       NaN  ...       NaN   
Hyundai    Elantra      NaN       NaN      NaN       NaN  ...       NaN   
Mazda      CX-5         NaN       NaN      NaN       NaN  ...       NaN   
Nissan     Altima       NaN       NaN      NaN       NaN  ...       NaN   
Skoda      Octavia      NaN  22600.00  5100.00   8594.59  ...   1700.00   
Tesla      Model 3      NaN       NaN      NaN       NaN  ...       NaN   
Toyota     Camry   19800.00       NaN      NaN    800.00  ...   7500.00   
           Corolla      NaN  54300.00      NaN   7800.00  ...       NaN   
Volkswagen Golf         NaN    500.00      NaN    300.00  ...       NaN   
           Jetta        NaN       NaN 46000.00    500.00  ...       NaN   
           Passat  16000.00   2000.00  8594.59       NaN  ...    800.00   
           Touareg      NaN       NaN      NaN       NaN  ...       NaN   

Year                    2013     2014      2015      2016      2017      2018  \
Make       Model                                                                
Chevrolet  Malibu        NaN      NaN       NaN    450.75       NaN       NaN   
Ford       Focus   292183.76 74083.76 171500.00 102789.17 102894.59  99500.00   
           Mondeo   41100.00      NaN       NaN       NaN   8600.00       NaN   
Hyundai    Elantra       NaN      NaN       NaN       NaN       NaN       NaN   
Mazda      CX-5          NaN      NaN       NaN       NaN       NaN    210.00   
Nissan     Altima        NaN      NaN       NaN       NaN       NaN       NaN   
Skoda      Octavia  11800.00 52600.00  16394.59  35700.00   2400.00 153200.00   
Tesla      Model 3       NaN      NaN       NaN       NaN       NaN       NaN   
Toyota     Camry         NaN      NaN       NaN   8594.59       NaN       NaN   
           Corolla       NaN      NaN   7500.00       NaN   9200.00       NaN   
Volkswagen Golf          NaN 13900.00   5300.00       NaN       NaN   5000.00   
           Jetta         NaN      NaN       NaN       NaN       NaN       NaN   
           Passat    1600.00  3800.00       NaN       NaN       NaN       NaN   
           Touareg       NaN      NaN       NaN       NaN       NaN       NaN   

Year                 2019  2021  2022  
Make       Model                       
Chevrolet  Malibu     NaN   NaN   NaN  
Ford       Focus      NaN   NaN   NaN  
           Mondeo     NaN   NaN   NaN  
Hyundai    Elantra   

## Save both the fines and owners dataframes to CSV files without an index

In [39]:
fines.to_csv("../data/fines.csv", index=False)
owners.to_csv("../data/owners.csv", index=False)

## review

In [40]:
concat_rows.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        914
dtype: int64

In [41]:
fines.count()

CarNumber    930
Refund       930
Fines        930
Make         930
Model        919
Year         930
dtype: int64

In [42]:
len(fines)

930

In [43]:
print(f"inner_merge: {inner_merge.shape}")
print(f"outer_merge: {outer_merge.shape}")
print(f"left_merge:  {left_merge.shape}")
print(f"right_merge: {right_merge.shape}")

inner_merge: (903, 7)
outer_merge: (933, 7)
left_merge:  (930, 7)
right_merge: (906, 7)
